# Day 2 — Manipulating & Cleaning Data
### Python for Data Analytics

Yesterday we loaded data. Today we make it **trustworthy and useful**: select
the rows we want, fix the messes real data always has, and summarise it.

**Agenda**
1. Select, filter, sort — your Sheets/Excel moves, in pandas
2. Cleaning — names, types, missing values, duplicates, messy text
3. Aggregation & grouping — pivot tables in one line
4. The project — clean a real Kaggle dataset in your own repo (Codespaces)

Run each cell with **Shift + Enter**, in order. 🏋️ **Your turn** cells are for
you to fill in.

---
## Our messy dataset

Real data is never tidy. Here's a small orders table with the problems you meet
all the time. We'll explore it, then fix every issue — the same arc as today's
project.

In [62]:
import pandas as pd

raw = pd.DataFrame({
    "Order ID":   ["A-1","A-2","A-3","A-4","A-5","A-6","A-7","A-8","A-9","A-10","A-11","A-12","A-3","A-6"],
    "Region":     ["West","East","East","Central","West","Central","East","West","Central","East","West","Central","East","Central"],
    "Category":   ["Furniture"," furniture","OFFICE","Office","Technology"," technology","Furniture","Office","Technology","Office","Furniture","Technology","OFFICE"," technology"],
    "Sales":      ["120.5","90","60","45.0","300","150.75","80","200","75.25","110","95","60","60","150.75"],
    "Units":      [2, None, 1, 3, 1, 2, None, 4, 1, 2, 3, 1, 1, 2],
    "Order Date": ["1/5/2023","2/8/2023","3/2/2023","4/9/2023","5/1/2023","6/3/2023","7/7/2023","8/8/2023","9/9/2023","10/10/2023","11/11/2023","12/12/2023","3/2/2023","6/3/2023"],
})
raw

,Order ID,Region,Category,Sales,Units,Order Date
0,A-1,West,Furniture,120.5,2.0,1/5/2023
1,A-2,East,furniture,90,NaN,2/8/2023
2,A-3,East,OFFICE,60,1.0,3/2/2023
3,A-4,Central,Office,45.0,3.0,4/9/2023
4,A-5,West,Technology,300,1.0,5/1/2023
5,A-6,Central,technology,150.75,2.0,6/3/2023
6,A-7,East,Furniture,80,NaN,7/7/2023
7,A-8,West,Office,200,4.0,8/8/2023
8,A-9,Central,Technology,75.25,1.0,9/9/2023
9,A-10,East,Office,110,2.0,10/10/2023


The first thing you do with *any* new data: look at it. `.info()` shows the
columns, their types, and how many values are filled in.

In [63]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Order ID    14 non-null     str    
 1   Region      14 non-null     str    
 2   Category    14 non-null     str    
 3   Sales       14 non-null     str    
 4   Units       12 non-null     float64
 5   Order Date  14 non-null     str    
dtypes: float64(1), str(5)
memory usage: 804.0 bytes


Spot the problems already:
- `Sales` is stored as **text** (`object`), not numbers — we can't add it up yet.
- `Units` has **missing** values (fewer non-null than rows).
- `Order Date` is **text**, not a date.
- `Category` is written inconsistently (`OFFICE`, `Office`, ` furniture`).
- Some rows look **duplicated** (`A-3`, `A-6` appear twice).

🏋️ **Your turn.** Print the **shape** of `raw` (rows, columns) and the list of
its column names.

In [64]:
# TODO: print raw.shape and list(raw.columns)

---
## Block 1 — Select, filter, sort

The everyday spreadsheet moves: pick columns, keep matching rows, order them.

### Selecting columns

One column (a **Series**) uses single brackets. Several columns (a
**DataFrame**) use double brackets.

In [65]:
raw["Region"]            # one column -> a Series

0        West
1        East
2        East
3     Central
4        West
5     Central
6        East
7        West
8     Central
9        East
10       West
11    Central
12       East
13    Central
Name: Region, dtype: str

In [66]:
raw[["Order ID", "Sales"]]   # several columns -> a DataFrame (note the double [[ ]])

,Order ID,Sales
0,A-1,120.5
1,A-2,90
2,A-3,60
3,A-4,45.0
4,A-5,300
5,A-6,150.75
6,A-7,80
7,A-8,200
8,A-9,75.25
9,A-10,110


### Filtering rows

Put a condition inside the brackets to keep only the rows where it's `True` —
just like a spreadsheet filter.

In [67]:
raw[raw["Region"] == "West"]

,Order ID,Region,Category,Sales,Units,Order Date
0,A-1,West,Furniture,120.5,2.0,1/5/2023
4,A-5,West,Technology,300,1.0,5/1/2023
7,A-8,West,Office,200,4.0,8/8/2023
10,A-11,West,Furniture,95,3.0,11/11/2023


Combine conditions with `&` (and) / `|` (or). **Each condition needs its own
parentheses.**

In [68]:
raw[(raw["Region"] == "West") & (raw["Category"] == "Furniture")]

,Order ID,Region,Category,Sales,Units,Order Date
0,A-1,West,Furniture,120.5,2.0,1/5/2023
10,A-11,West,Furniture,95,3.0,11/11/2023


In [69]:
# Match any of several values with .isin(...)
raw[raw["Region"].isin(["West", "East"])]

,Order ID,Region,Category,Sales,Units,Order Date
0,A-1,West,Furniture,120.5,2.0,1/5/2023
1,A-2,East,furniture,90,NaN,2/8/2023
2,A-3,East,OFFICE,60,1.0,3/2/2023
4,A-5,West,Technology,300,1.0,5/1/2023
6,A-7,East,Furniture,80,NaN,7/7/2023
7,A-8,West,Office,200,4.0,8/8/2023
9,A-10,East,Office,110,2.0,10/10/2023
10,A-11,West,Furniture,95,3.0,11/11/2023
12,A-3,East,OFFICE,60,1.0,3/2/2023


🏋️ **Your turn.** Keep only the rows where `Region` is `"Central"`, and show
just the `Order ID` and `Sales` columns.

In [70]:
# TODO: filter to Region == "Central", then select Order ID and Sales

### Sorting

`sort_values` orders rows. You can sort by one column or several.

In [71]:
raw.sort_values("Order ID")

,Order ID,Region,Category,Sales,Units,Order Date
0,A-1,West,Furniture,120.5,2.0,1/5/2023
9,A-10,East,Office,110,2.0,10/10/2023
10,A-11,West,Furniture,95,3.0,11/11/2023
11,A-12,Central,Technology,60,1.0,12/12/2023
1,A-2,East,furniture,90,NaN,2/8/2023
2,A-3,East,OFFICE,60,1.0,3/2/2023
12,A-3,East,OFFICE,60,1.0,3/2/2023
3,A-4,Central,Office,45.0,3.0,4/9/2023
4,A-5,West,Technology,300,1.0,5/1/2023
5,A-6,Central,technology,150.75,2.0,6/3/2023


In [72]:
# Sort by Region first, then Order ID within each region.
raw.sort_values(["Region", "Order ID"])

,Order ID,Region,Category,Sales,Units,Order Date
11,A-12,Central,Technology,60,1.0,12/12/2023
3,A-4,Central,Office,45.0,3.0,4/9/2023
5,A-6,Central,technology,150.75,2.0,6/3/2023
13,A-6,Central,technology,150.75,2.0,6/3/2023
8,A-9,Central,Technology,75.25,1.0,9/9/2023
9,A-10,East,Office,110,2.0,10/10/2023
1,A-2,East,furniture,90,NaN,2/8/2023
2,A-3,East,OFFICE,60,1.0,3/2/2023
12,A-3,East,OFFICE,60,1.0,3/2/2023
6,A-7,East,Furniture,80,NaN,7/7/2023


🏋️ **Your turn.** Sort `raw` by `Region`, but in **descending** order.
(Hint: `sort_values("Region", ascending=False)`.)

In [73]:
# TODO: sort by Region descending

### Picking by label vs position — `loc` and `iloc`

Two precise ways to grab specific rows and columns:
- **`.loc[]`** selects by **label** — the row's index and the **column name**.
- **`.iloc[]`** selects by **integer position** — plain row/column numbers.

In [74]:
raw.loc[0]                          # the row whose index label is 0 (a Series)

Order ID            A-1
Region             West
Category      Furniture
Sales             120.5
Units               2.0
Order Date     1/5/2023
Name: 0, dtype: object

In [75]:
raw.loc[0, "Region"]                # a single cell: row label 0, column "Region"

'West'

In [76]:
raw.loc[0:2, ["Region", "Sales"]]   # label slicing is INCLUSIVE -> rows 0, 1 AND 2

,Region,Sales
0,West,120.5
1,East,90
2,East,60


In [77]:
raw.iloc[0]                         # the FIRST row, by position

Order ID            A-1
Region             West
Category      Furniture
Sales             120.5
Units               2.0
Order Date     1/5/2023
Name: 0, dtype: object

In [78]:
raw.iloc[0:2, 0:2]                  # position slicing is end-EXCLUSIVE -> rows 0,1 & cols 0,1

,Order ID,Region
0,A-1,West
1,A-2,East


💡 Rule of thumb: use **`loc`** when you know the **name/label**, **`iloc`** when
you know the **position**. The classic gotcha: `loc[0:2]` *includes* row 2, but
`iloc[0:2]` stops *before* it.

🏋️ **Your turn.** Use `iloc` to grab the **last row** of `raw`.
(Hint: the last position is `-1`.)

In [79]:
# TODO: get the last row of raw using iloc

---
## Block 2 — Cleaning

Now we fix the mess, one issue at a time. Each fix below is exactly what you'll
do on the real dataset in the project.

### 1. Tidy the column names

`"Order ID"` is awkward to type (spaces, capitals). Standardise to `order_id`
style — lower case, no spaces.

In [80]:
raw.columns = raw.columns.str.strip().str.lower().str.replace(" ", "_")
list(raw.columns)

['order_id', 'region', 'category', 'sales', 'units', 'order_date']

📝 From here on the columns are `order_id`, `region`, `category`, `sales`,
`units`, `order_date`.

### 2. Fix the types

`to_numeric` turns text into numbers; `to_datetime` turns text into real dates.

In [81]:
raw["sales"] = pd.to_numeric(raw["sales"])
raw["order_date"] = pd.to_datetime(raw["order_date"], format="%m/%d/%Y")
raw.dtypes

order_id                 str
region                   str
category                 str
sales                float64
units                float64
order_date    datetime64[us]
dtype: object

💡 If some values can't be converted (e.g. `"n/a"`), add `errors="coerce"` and
pandas turns the bad ones into `NaN` instead of crashing:

In [82]:
pd.to_numeric(pd.Series(["10", "20", "n/a"]), errors="coerce")

0    10.0
1    20.0
2     NaN
dtype: float64

🏋️ **Your turn.** Convert this price Series to numbers, turning the bad value
into `NaN`: `pd.Series(["1.99", "3.50", "free"])`.

In [83]:
# TODO: pd.to_numeric(..., errors="coerce")

### 3. Handle missing values

See where the gaps are, then decide: **fill** them or **drop** them.

In [84]:
raw.isna().sum()

order_id      0
region        0
category      0
sales         0
units         2
order_date    0
dtype: int64

In [85]:
# Option A: fill the missing Units with 0.
raw["units"] = raw["units"].fillna(0)

# Option B (not used here): raw.dropna(subset=["units"]) would drop those rows.
raw["units"].isna().sum()

np.int64(0)

🏋️ **Your turn.** Imagine `s = pd.Series([10, None, 30])`. Fill the missing
value with the **mean** of the series. (Hint: `s.fillna(s.mean())`.)

In [86]:
# TODO: fill the missing value with the mean
s = pd.Series([10, None, 30])

### 4. Remove duplicates

`duplicated()` flags repeated rows; `drop_duplicates()` removes them.

In [87]:
print("duplicate rows:", raw.duplicated().sum())
raw = raw.drop_duplicates()
print("rows now:", len(raw))

duplicate rows: 2
rows now: 12


💡 You can also dedupe on *specific* columns: `drop_duplicates(subset=["order_id"])`
keeps one row per order id.

### 5. Clean up messy text

`category` has `"OFFICE"`, `"Office"`, `" furniture"` — the same things written
differently. Strip spaces and standardise the case.

In [88]:
raw["category"] = raw["category"].str.strip().str.title()
raw["category"].value_counts()

category
Furniture     4
Office        4
Technology    4
Name: count, dtype: int64

💡 For known one-off fixes, `replace` maps specific values to new ones, e.g.
`df["region"].replace({"W": "West", "E": "East"})`.

🏋️ **Your turn.** Normalise this Series so all three become `"Yes"`:
`pd.Series(["YES", "yes", " Yes "])`. (Hint: `.str.strip().str.title()`.)

In [89]:
# TODO: normalise to all "Yes"


---
## Block 3 — Aggregation & grouping

Our data is clean — now we summarise it. `value_counts` counts categories;
`groupby` is the pivot table.

In [90]:
raw["region"].value_counts()

region
West       4
East       4
Central    4
Name: count, dtype: int64

In [91]:
# Add normalize=True to get proportions instead of counts.
raw["region"].value_counts(normalize=True)

region
West       0.333333
East       0.333333
Central    0.333333
Name: proportion, dtype: float64

### groupby — the pivot table

Split the data into groups, then compute something per group.

In [92]:
raw.groupby("region")["sales"].sum()          # total sales per region

region
Central    331.0
East       340.0
West       715.5
Name: sales, dtype: float64

In [93]:
# Several statistics at once.
raw.groupby("region")["sales"].agg(["sum", "mean", "count"])

,sum,mean,count
region,,,
Central,331.0,82.750,4
East,340.0,85.000,4
West,715.5,178.875,4


In [94]:
# Group by two columns: sales per region AND category.
raw.groupby(["region", "category"])["sales"].sum()

region   category  
Central  Office         45.0
         Technology    286.0
East     Furniture     170.0
         Office        170.0
West     Furniture     215.5
         Office        200.0
         Technology    300.0
Name: sales, dtype: float64

### pivot_table — a 2-D summary

`pivot_table` lays groups out as a grid — regions down the side, categories
across the top — exactly like a spreadsheet pivot table.

In [ ]:
pd.pivot_table(raw, values="sales", index="region", columns="category",
               aggfunc="sum", fill_value=0)

🏋️ **Your turn.** Compute the **average** `sales` per `category`.
(Hint: `groupby("category")` then `.mean()` on `sales`.)

In [ ]:
# TODO: average sales per category

🏋️ **Your turn.** Compute the **total `units`** sold per `region`, and sort the
result from highest to lowest.

In [ ]:
# TODO: total units per region, sorted descending

---
## Block 4 — The project: your first week at Superstore 🕵️

**The scenario:** it's your first week as a data analyst at *Superstore*.
Leadership just dropped a messy data export on your desk and a list of
questions. Your job: clean it, summarise it, and brief them — in your **own
repository**, using **GitHub Codespaces**, just like a real data team.

**The dataset:** *Sample - Superstore* (retail orders) — already bundled in
`data/`; it originally comes from Kaggle (see `data/README.md`).
**Setup:** create your repo from the template, open it in Codespaces, and go.

Then work through, filling in the `# TODO`s:

| Step | File | Your mission |
|------|------|--------------|
| 0 | `src/explore.py` | size up the mess (nothing to fill in) |
| 1 | `src/clean.py` | encoding, names, dates, drop columns, duplicates, missing values |
| 2 | `src/transform.py` | new columns, group & summarise, save `summary_*.csv` |
| 3 | `src/questions.py` | answer leadership's questions with code |
| 🎁 | `notebooks/02_predict.ipynb` | **bonus:** chart your data, and let Python *predict* with it |

Everything you need is in the blocks above. Finish by **committing your
`output/` files** — you'll have shipped a small, reproducible pipeline. 🚀